# Mounting Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Config-Holds all the hyperparameter settings and paths for the pipeline.

In [ ]:
"""
config.py
Holds all the hyperparameter settings and paths for the pipeline.
"""

# We keep all our main settings here so we don't have to hunt for them later.
# Tweak the batch_size or num_workers depending on what your GPU/CPU can handle.
CONFIG = {
    # -- Training Settings --
    "num_epochs":         25,
    "batch_size":         32,
    "lr":                 1e-4,
    "weight_decay":       1e-4,
    "n_folds":            5,
    "num_workers":        2,
    "seed":               42,

    # Make sure your Google Drive is mounted if you are running this in Colab!
    "out_dir":            "/content/drive/MyDrive/mybrain_research",

    # -- Dataset specific settings --
    "slices_per_patient": 10,

    # -- Cache files --
    # Preprocessing takes forever, so we save the results to these files.
    "brisc_cache_path":    "brisc_cache.pkl",
    "figshare_cache_path": "figshare_cache.pkl",

    # -- Run Mode --
    # Change 'skip_training' to False if you want to train from scratch.
    # Otherwise, it'll just load the weights from the path below.
    "skip_training": False,
    "v3_model_path": "/content/drive/MyDrive/Brain_Tumor_Research/best_model_brisc.pth"
}

# Supported image types
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Preprocessing

In [ ]:
"""
preprocessing.py
Handles image cleaning, cropping, enhancements, and caching so we don't bottleneck the CPU.
"""

import os
import cv2
import pickle
import numpy as np
from tqdm import tqdm

def preprocess_image(image_path: str, output_size: int = 224):
    """
    Cleans up the MRI scans before feeding them to the model.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Auto-crop: There's usually a lot of useless black space around the brain in MRIs.
    # We use contours to find the actual brain and crop out the empty void.
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh    = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts    = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        pad        = 2
        img = img[max(0, y - pad): y + h + pad, max(0, x - pad): x + w + pad]

    # 2. CLAHE (Contrast Limited Adaptive Histogram Equalization):
    # This basically enhances the contrast of the image to make the tumor features pop out more.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img   = clahe.apply(img)

    # 3. Normalize using Z-score and scale back to standard 0-255 pixel values
    f   = img.astype(np.float32)
    f   = (f - f.mean()) / (f.std() + 1e-8)
    f   = (f - f.min()) / (f.max() - f.min() + 1e-8)
    img = (f * 255).astype(np.uint8)

    # 4. Resize to exactly what ResNet expects (224x224) and fake it into a 3-channel RGB image
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    return img

def build_cache(records, cache_path: str = ""):
    """
    Preprocessing thousands of images on the fly during training slows everything down.
    This function processes them once and saves them into a dictionary (cache).
    """
    if cache_path and os.path.exists(cache_path):
        print(f"Awesome, found an existing cache at {cache_path}. Loading it up...")
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)

        # Just in case we added new images since the last run
        missing = [r["path"] for r in records if r["path"] not in cache]
        if missing:
            for path in tqdm(missing, desc="Updating cache with new images", unit="img"):
                img = preprocess_image(path)
                cache[path] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)
            with open(cache_path, "wb") as f:
                pickle.dump(cache, f)
        return cache

    print(f"Building cache for {len(records)} images. Go grab a coffee, this only runs once...")
    cache = {}
    for r in tqdm(records, desc="Preprocessing images", unit="img"):
        img = preprocess_image(r["path"])
        # If an image is corrupted, just throw in a blank black image so the code doesn't crash
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f)

    return cache

# Data Handlers - Everything related to downloading datasets, parsing messy filenames, and feeding data to PyTorch.

In [ ]:
"""
data_handlers.py
Downloads the data, parses the complex filenames to extract patient info, and creates the PyTorch Dataset.
"""

import os
import re
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T

# If you are splitting files, you'd import the config and preprocessing here:
# from config import IMG_EXTS
# from preprocessing import preprocess_image

def download_datasets():
    # We use kagglehub to pull the datasets straight into our environment
    try:
        import kagglehub
    except ImportError:
        print("Oops! You need to install kagglehub first. Run: pip install kagglehub")
        sys.exit(1)

    print("Downloading BRISC 2025 dataset...")
    brisc_path = kagglehub.dataset_download("briscdataset/brisc2025")

    print("Downloading Figshare dataset...")
    figshare_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

    return brisc_path, figshare_path

# Maps the short codes in the BRISC filenames to readable tumor types
BRISC_TYPE_MAP = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}

# Regex to rip apart the BRISC filenames so we can extract the slice ID and tumor type
_BRISC_RE = re.compile(r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE)

def parse_brisc_filename(filename: str, slices_per_patient: int):
    stem = Path(filename).stem
    m    = _BRISC_RE.match(stem)
    if not m:
        return None

    slice_id  = int(m.group(2))
    type_code = m.group(3).lower()
    label     = 0 if type_code == "no" else 1

    # Grouping slices into buckets so we can assign them a dummy "Patient ID".
    # This prevents the model from seeing the same patient's brain in both train and validation!
    patient_bucket = slice_id // slices_per_patient
    patient_id     = f"{type_code}_{patient_bucket:04d}"

    return {
        "slice_id":   slice_id,
        "type_code":  type_code,
        "type_name":  BRISC_TYPE_MAP.get(type_code, type_code),
        "label":      label,
        "patient_id": patient_id,
        "split":      m.group(1).lower(),
    }

def load_brisc(brisc_root: str, slices_per_patient: int = 10):
    # Digs through the downloaded BRISC folder and builds a list of dictionaries for every image
    class_root = os.path.join(brisc_root, "brisc2025", "classification_task")
    if not os.path.isdir(class_root):
        class_root = os.path.join(brisc_root, "classification_task")

    records, skipped = [], 0

    for dirpath, _, filenames in os.walk(class_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            parsed = parse_brisc_filename(fname, slices_per_patient)
            if parsed is None:
                skipped += 1
                continue

            records.append({
                "path":       os.path.join(dirpath, fname),
                "label":      parsed["label"],
                "patient_id": parsed["patient_id"],
                "type_name":  parsed["type_name"],
                "slice_id":   parsed["slice_id"],
            })

    print(f"BRISC Dataset loaded: Found {len(records)} images.")
    return records

def load_figshare(figshare_root: str):
    # Simpler parser for the Figshare dataset. It just checks the folder name to get the label.
    records = []
    for dirpath, _, filenames in os.walk(figshare_root):
        class_folder = os.path.basename(dirpath)
        if class_folder.lower() in {"training", "testing", "train", "test", "brain-tumor-mri-dataset", ""}:
            continue

        label = 0 if class_folder.lower() in NEGATIVE_FOLDERS else 1

        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            full_path = os.path.join(dirpath, fname)

            # Generate a fake patient ID based on the filename numbers
            tokens = re.findall(r'\d+', Path(fname).stem)
            pid = f"fg_{tokens[-1].zfill(6)}" if tokens else f"fg_{abs(hash(Path(fname).stem)) % 1_000_000:06d}"

            records.append({"path": full_path, "label": label, "patient_id": pid})

    print(f"Figshare Dataset loaded: Found {len(records)} images.")
    return records

def build_transforms(augment: bool):
    # Standard ImageNet stats for ResNet
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    ops = [T.ToPILImage()]

    # If we are training, throw in some random flips and rotations to prevent overfitting
    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ]

    ops += [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return T.Compose(ops)

class BrainMRIDataset(Dataset):
    """ PyTorch wrapper so the DataLoader knows how to fetch our images and labels """
    def __init__(self, records, transform, cache=None):
        self.records   = records
        self.transform = transform
        self.cache     = cache

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        path = rec["path"]

        # Grab from cache if we have it, otherwise process it live
        if self.cache is not None and path in self.cache:
            img = self.cache[path]
        else:
            img = preprocess_image(path)
            if img is None: img = np.zeros((224, 224, 3), dtype=np.uint8)

        return self.transform(img), rec["label"]

# Models and Metrics - Defines the ResNet18 model, calculates accuracy/AUC, and generates fancy plots.

In [ ]:
"""
model_and_metrics.py
Defines the ResNet18 model, calculates accuracy/AUC, and generates fancy plots.
"""

import os
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import matplotlib
matplotlib.use("Agg") # Prevents weird display bugs when running on servers/Colab
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

def build_model(pretrained: bool = True):
    # Grab the standard ResNet18 architecture
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.resnet18(weights=weights)

    # ResNet normally outputs 1000 classes. We rip off the final layer and replace
    # it with a single node (0 or 1) since we are just doing binary classification (Tumor vs No Tumor).
    in_feat = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5), # Helps prevent the model from memorizing the training data
        nn.Linear(in_feat, 1)
    )
    return model

def compute_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float = 0.5):
    # Compares the model's predictions against the actual ground truth
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": float(tn / (tn + fp + 1e-8)),
        "f1":          f1_score(labels, preds, zero_division=0),
        "auc_roc":     roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
    }

def print_metrics(m, header: str = ""):
    # Just a clean way to print the metrics to the console
    bar = "=" * 52
    print(f"\n{bar}\n  {header}\n{bar}")
    for k, v in m.items():
        print(f"  {k:<14}: {v:.4f}" if isinstance(v, float) else f"  {k:<14}: {v}")
    print(bar)

def plot_curves(history, fold: int, out_dir: str):
    # Plots training vs validation loss so we can see if the model is overfitting
    os.makedirs(out_dir, exist_ok=True)
    ep = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ep, history["train_loss"], label="Train"); axes[0].plot(ep, history["val_loss"], label="Val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(ep, history["train_acc"],  label="Train"); axes[1].plot(ep, history["val_acc"],  label="Val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"fold{fold}_curves.png"), dpi=120)
    plt.close()

def plot_cm(labels, probs, title, out_dir, fname, threshold=0.5):
    # Generates a heatmap showing true positives, false positives, etc.
    os.makedirs(out_dir, exist_ok=True)
    preds = (probs >= threshold).astype(int)
    cm    = confusion_matrix(labels, preds, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Non-Tumor", "Tumor"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), dpi=120)
    plt.close()

def plot_roc(labels, probs, title, out_dir, fname):
    # Generates the Receiver Operating Characteristic curve
    if len(np.unique(labels)) < 2: return
    os.makedirs(out_dir, exist_ok=True)

    fpr, tpr, _ = roc_curve(labels, probs)
    auc         = roc_auc_score(labels, probs)

    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title); plt.legend(); plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), dpi=120)
    plt.close()

# Main Function - The master script. Executes the cross-validation loops and final external testing.

In [ ]:
"""
main.py
The master script. Executes the cross-validation loops and final external testing.
"""

import os
import copy
import json
import random
import logging
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# If splitting files, import everything needed here:
# from config import CONFIG
# from preprocessing import build_cache
# from data_handlers import download_datasets, load_brisc, load_figshare, build_transforms, BrainMRIDataset
# from model_and_metrics import build_model, compute_metrics, print_metrics, plot_curves, plot_cm, plot_roc

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
log = logging.getLogger(__name__)

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # We use autocast (mixed precision) to train faster and use less VRAM
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * imgs.size(0)
        preds     = (torch.sigmoid(logits) >= 0.5).long()
        correct  += (preds == labels.long()).sum().item()
        total    += imgs.size(0)

    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        probs     = torch.sigmoid(logits)
        loss_sum += loss.item() * imgs.size(0)
        correct  += ((probs >= 0.5).long() == labels.long()).sum().item()
        total    += imgs.size(0)

        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())

    return (loss_sum / total, correct / total, np.array(all_labels), np.array(all_probs))

def run_cv(records, device, config, brisc_cache=None):
    labels_arr = np.array([r["label"] for r in records])
    groups_arr = np.array([r["patient_id"] for r in records])

    # We use StratifiedGroupKFold so all slices from a single patient stay in the SAME fold.
    # Otherwise, the model basically memorizes the patient's brain and cheats on the validation set!
    sgkf = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])

    fold_results = []
    best_auc, best_model = -1.0, None

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(records, labels_arr, groups=groups_arr), start=1):
        log.info(f"\n{'='*58}\n  FOLD {fold}/{config['n_folds']}  |  train={len(tr_idx)}  val={len(vl_idx)}\n{'='*58}")

        train_recs = [records[i] for i in tr_idx]
        val_recs   = [records[i] for i in vl_idx]

        train_ds = BrainMRIDataset(train_recs, build_transforms(augment=True), cache=brisc_cache)
        val_ds   = BrainMRIDataset(val_recs, build_transforms(augment=False), cache=brisc_cache)

        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], pin_memory=(device.type == "cuda"), drop_last=True)
        val_dl   = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=(device.type == "cuda"))

        model     = build_model().to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"])
        scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        best_vl_auc, best_wts = -1.0, None

        for epoch in range(1, config["num_epochs"] + 1):
            tl, ta = train_one_epoch(model, train_dl, criterion, optimizer, scaler, device)
            val_loss, val_acc, val_labels, val_probs = evaluate(model, val_dl, criterion, device)
            scheduler.step()

            ep_auc = roc_auc_score(val_labels, val_probs) if len(np.unique(val_labels)) > 1 else 0.0
            history["train_loss"].append(tl); history["train_acc"].append(ta)
            history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)

            log.info(f"  Ep {epoch:02d}/{config['num_epochs']} | TrL={tl:.4f} TrA={ta:.4f} | VlL={val_loss:.4f} VlA={val_acc:.4f} AUC={ep_auc:.4f}")

            # Keep track of the best performing weights in this fold
            if ep_auc > best_vl_auc:
                best_vl_auc = ep_auc
                best_wts    = copy.deepcopy(model.state_dict())

        # Testing the best weights for this fold
        model.load_state_dict(best_wts)
        _, _, fl, fp = evaluate(model, val_dl, criterion, device)
        m = compute_metrics(fl, fp)
        m["fold"] = fold
        fold_results.append(m)

        print_metrics(m, header=f"Fold {fold} Final Results")
        plot_curves(history, fold, config["out_dir"])
        plot_cm(fl, fp, f"Fold {fold} Confusion Matrix", config["out_dir"], f"fold{fold}_cm.png")
        plot_roc(fl, fp, f"Fold {fold} ROC Curve", config["out_dir"], f"fold{fold}_roc.png")

        # Did this fold beat our absolute best model across all folds?
        if m["auc_roc"] > best_auc:
            best_auc   = m["auc_roc"]
            best_model = copy.deepcopy(model)
            torch.save(best_wts, os.path.join(config["out_dir"], "best_model_brisc.pth"))
            log.info(f"  ✓ New best model saved globally (AUC={best_auc:.4f})")

    return best_model, fold_results

def run_external_test(model, records, device, config, figshare_cache=None):
    # Tests our best BRISC model against a completely different dataset (Figshare) to see how it generalizes
    test_ds = BrainMRIDataset(records, build_transforms(augment=False), cache=figshare_cache)
    test_dl = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=(device.type == "cuda"))

    criterion         = nn.BCEWithLogitsLoss()
    _, _, labs, probs = evaluate(model, test_dl, criterion, device)
    m                 = compute_metrics(labs, probs)

    print_metrics(m, header="EXTERNAL TEST — Figshare (held-out)")
    plot_cm(labs, probs, "Figshare — Confusion Matrix", config["out_dir"], "figshare_cm.png")
    plot_roc(labs, probs, "Figshare — ROC Curve", config["out_dir"], "figshare_roc.png")

    json_path = os.path.join(config["out_dir"], "figshare_metrics.json")
    with open(json_path, "w") as f:
        json.dump({k: float(v) for k, v in m.items()}, f, indent=2)

    return m

def main():
    # Setup seeds so we can reproduce our results exactly next time
    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    random.seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])
    torch.manual_seed(CONFIG["seed"])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG["seed"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Using Device: {device}")

    # 1. Grab data
    brisc_path, figshare_path = download_datasets()
    brisc_records    = load_brisc(brisc_path, slices_per_patient=CONFIG["slices_per_patient"])
    figshare_records = load_figshare(figshare_path)

    if not brisc_records or not figshare_records:
        log.error("Something went wrong, missing records.")
        sys.exit(1)

    # 2. Build or load cache
    log.info("\n── Checking Preprocessing Caches ──")
    brisc_cache    = build_cache(brisc_records, cache_path=CONFIG["brisc_cache_path"])
    figshare_cache = build_cache(figshare_records, cache_path=CONFIG["figshare_cache_path"])

    # 3. Train or load model
    if CONFIG.get("skip_training") and os.path.exists(CONFIG["v3_model_path"]):
        log.info(f"\n>>> Skipping Training. Loading model from {CONFIG['v3_model_path']} <<<")
        best_model = build_model(pretrained=False).to(device)
        state_dict = torch.load(CONFIG["v3_model_path"], map_location=device)
        best_model.load_state_dict(state_dict)
        best_model.eval()
    else:
        log.info("\n>>> Starting 5-Fold Cross Validation Training <<<")
        best_model, cv_results = run_cv(brisc_records, device, CONFIG, brisc_cache=brisc_cache)

    # 4. Final test on Figshare
    log.info("\nRunning external validation on Figshare dataset...")
    run_external_test(best_model, figshare_records, device, CONFIG, figshare_cache=figshare_cache)

    log.info(f"\nPipeline complete! All plots and metrics are saved to -> {CONFIG['out_dir']}/")

if __name__ == "__main__":
    main()

100%|██████████| 250M/250M [00:08<00:00, 30.7MB/s]

Extracting files...


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
BRISC Dataset loaded: Found 6000 images.
Figshare Dataset loaded: Found 7200 images.
Building cache for 6000 images. Go grab a coffee, this only runs once...


Preprocessing images: 100%|██████████| 6000/6000 [00:30<00:00, 197.57img/s]


Building cache for 7200 images. Go grab a coffee, this only runs once...


Preprocessing images: 100%|██████████| 7200/7200 [01:31<00:00, 78.74img/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]



  Fold 1 Final Results
  accuracy      : 1.0000
  precision     : 1.0000
  recall        : 1.0000
  specificity   : 1.0000
  f1            : 1.0000
  auc_roc       : 1.0000
  fold          : 1

  Fold 2 Final Results
  accuracy      : 0.9992
  precision     : 1.0000
  recall        : 0.9990
  specificity   : 1.0000
  f1            : 0.9995
  auc_roc       : 1.0000
  fold          : 2


KeyboardInterrupt: 